# 03 - Features and pipeline (judged on June)

Every learned quantity (planned-distance proxy, zone crawl rates) is fitted on **May only** and applied unchanged to June. Inside May the quantities are computed **out-of-fold** so a training trip never sees its own label or distance.

In [1]:
import sys; sys.path.insert(0, "..")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path
from src.clean import *
from src.features import *
from src.models import *
from src.evaluate import *
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
FIG = Path("../reports/figures"); FIG.mkdir(parents=True, exist_ok=True)
df, audit = build_dataset()
train, val, test = [df[df.split == s].reset_index(drop=True) for s in ("train", "validation", "test")]


## 1. A pipeline that cannot leak: three tests

In [2]:
M, zh = build_matrix(train, {"val": val})
cols_used = set(ALL_FEATS) | {"PULocationID", "DOLocationID"}
# Test 1: no forbidden column is a model feature
assert not (set(ALL_FEATS) & set(FORBIDDEN)), "forbidden column in features"
# Test 2: transform ignores the target and distance of the data it transforms
v2 = val[["PULocationID", "DOLocationID", "trip_distance"]].copy(); v2["trip_distance"] = 999.0
assert zh.transform(v2).equals(zh.transform(val[["PULocationID", "DOLocationID", "trip_distance"]])), "transform reads trip_distance"
# Test 3: everything learned comes from May: tables only contain May's zone pairs
assert zh.tables_["pair"]["n"].sum() == len(train)
print("all three leak tests passed")
print("train/val feature shapes:", M["train"][ALL_FEATS].shape, M["val"][ALL_FEATS].shape)

all three leak tests passed
train/val feature shapes: (187612, 17) (186114, 17)


## 2. What each addition did on June
Logistic regression and a decision tree (depth 8, min 1,000 per leaf), PR AUC on June. Base rate is the PR AUC of a useless model.

In [3]:
ytr, yva = train.crawl.values, val.crawl.values
sets = {"time only": TIME_FEATS, "+ geography flags": TIME_FEATS + GEO_FEATS, "+ planned distance": TIME_FEATS + GEO_FEATS + DIST_FEATS,
        "+ zone crawl rates": TIME_FEATS + GEO_FEATS + DIST_FEATS + ["pu_rate", "do_rate"], "+ zone-pair rate": ALL_FEATS}
rows = []
for name, c in sets.items():
    lr = make_logreg(c).fit(M["train"][c], ytr); tr_ = make_tree(8, 1000).fit(M["train"][c], ytr)
    rows.append({"feature set": name, "n_features": len(c),
                 "LogReg PR AUC": metrics(yva, lr.predict_proba(M["val"][c])[:, 1], .3)["pr_auc"],
                 "Tree PR AUC": metrics(yva, tr_.predict_proba(M["val"][c])[:, 1], .3)["pr_auc"]})
ladder = pd.DataFrame(rows).set_index("feature set").round(3); print(f"June base rate (PR AUC of a useless model): {yva.mean():.3f}"); ladder

June base rate (PR AUC of a useless model): 0.185


,n_features,LogReg PR AUC,Tree PR AUC
feature set,,,
time only,4,0.321,0.327
+ geography flags,12,0.368,0.366
+ planned distance,14,0.433,0.425
+ zone crawl rates,16,0.592,0.585
+ zone-pair rate,17,0.619,0.605


## 3. Features that lost, and the upper bound
Two reported on purpose: (a) `is_flex`, which adds nothing useful and whose payment type is only settled at the end; (b) the measured `trip_distance`, an **oracle that is not allowed at pickup**. Compare the oracle tree with the tree row above it: the real measured distance adds almost nothing over our zone-pair estimate, so the proxy captures what the planned distance would give.

In [4]:
c = ALL_FEATS
Xtr, Xva = M["train"][c].copy(), M["val"][c].copy()
res = {}
res["chosen features (LogReg)"] = metrics(yva, make_logreg(c).fit(Xtr, ytr).predict_proba(Xva)[:, 1], .3)["pr_auc"]
res["chosen features (tree depth 8, leaf 1000)"] = metrics(yva, make_tree(8, 1000).fit(Xtr, ytr).predict_proba(Xva)[:, 1], .3)["pr_auc"]
Xtr_f, Xva_f = Xtr.assign(is_flex=train.is_flex.values), Xva.assign(is_flex=val.is_flex.values)
res["+ is_flex (rejected: payment type is settled at the end)"] = metrics(yva, make_logreg(c + ["is_flex"]).fit(Xtr_f, ytr).predict_proba(Xva_f)[:, 1], .3)["pr_auc"]
Xtr_o, Xva_o = Xtr.assign(measured_dist=np.log1p(train.trip_distance.values)), Xva.assign(measured_dist=np.log1p(val.trip_distance.values))
res["ORACLE tree (depth 8, leaf 1000) + measured trip_distance (forbidden)"] = metrics(yva, make_tree(8, 1000).fit(Xtr_o, ytr).predict_proba(Xva_o)[:, 1], .3)["pr_auc"]
pd.Series(res).round(3).to_frame("June PR AUC")

,June PR AUC
chosen features (LogReg),0.619
"chosen features (tree depth 8, leaf 1000)",0.605
+ is_flex (rejected: payment type is settled at the end),0.619
"ORACLE tree (depth 8, leaf 1000) + measured trip_distance (forbidden)",0.608


## 4. Tuning the tree on June
Only depth and minimum leaf size are tuned, only on June.

In [5]:
g = []
for d in (2, 3, 4, 6, 8, 10, 12):
    for leaf in (200, 1000, 3000):
        p = make_tree(d, leaf).fit(Xtr, ytr).predict_proba(Xva)[:, 1]; g.append({"depth": d, "min_leaf": leaf, "PR AUC": metrics(yva, p, .3)["pr_auc"]})
g = pd.DataFrame(g).pivot(index="depth", columns="min_leaf", values="PR AUC").round(3); g

min_leaf,200,1000,3000
depth,,,
2,0.447,0.447,0.447
3,0.494,0.494,0.497
4,0.539,0.539,0.535
6,0.583,0.581,0.574
8,0.609,0.605,0.583
10,0.622,0.612,0.583
12,0.628,0.613,0.583


In [6]:
best = g.stack().idxmax(); print("best tree (depth, min_leaf):", best, "PR AUC", round(g.stack().max(), 3))
Cs = {C: metrics(yva, make_logreg(ALL_FEATS, C).fit(Xtr, ytr).predict_proba(Xva)[:, 1], .3)["pr_auc"] for C in (0.01, 0.1, 1, 10)}
print("LogReg C vs June PR AUC:", {k: round(v, 4) for k, v in Cs.items()})

best tree (depth, min_leaf): (np.int64(12), np.int64(200)) PR AUC 0.628


LogReg C vs June PR AUC: {0.01: 0.6177, 0.1: 0.619, 1: 0.6191, 10: 0.6193}


**Conclusion.** The biggest gains come from the planned-distance proxy and the learned zone and zone-pair crawl rates; time and geography flags alone are weak. Notebook 04 compares the tuned models and finishes the evaluation.